# 03 — Evaluate six planner conditions

Three base-model strategies (always ONCE, always SPLIT, prompted adaptive) and three fine-tuned-model conditions (forced ONCE, forced SPLIT, adaptive). All use the same `run_evaluation()` pipeline. Forced conditions are constrained ablations, not the model's natural adaptive policy.


## 1. Configure paths and execution

In [ ]:
from pathlib import Path

GITHUB_REPO = "RanaShoaaib/Adaptive-RAG"
GITHUB_REF = "main"  # Prefer the same commit as notebook 02
INSTALL_FROM_GITHUB = True
LOCAL_REPO = "/content/Adaptive-RAG"
MOUNT_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive/Adaptive-RAG")
TEST_PATH = None  # Defaults to the tracked test subset. Set a Path to override.
ADAPTER_DIR = DRIVE_ROOT / "results/adaptive_planner_v1/adapter"
RESULTS_DIR = DRIVE_ROOT / "results/heldout_evaluation"

RUN_BASELINES = False  # Base model: always_once, always_split, prompted
RUN_ADAPTIVE = False   # Fine-tuned model: adaptive_once, adaptive_split, adaptive
# Enable either group after a small pilot; each run incurs retrieval/answering cost.
ADAPTIVE_USE_4BIT = True
EXPERIMENT_PREFIX = "test_v1"
RUN_VALIDATION_CURVE = False  # Run adaptive_once/adaptive_split on the validation subset, then choose tau from the quality-cost curve
GAIN_FRACTION = 0.8  # Fix BEFORE looking at the curve: smallest SPLIT share that captures this fraction of the always-SPLIT recall gain


## 2. Install the package

In [ ]:
import subprocess, sys

if INSTALL_FROM_GITHUB:
    checkout = Path(LOCAL_REPO)
    if not checkout.exists():
        subprocess.check_call(["git", "clone", f"https://github.com/{GITHUB_REPO}.git", str(checkout)])
    subprocess.check_call(["git", "-C", str(checkout), "fetch", "origin", GITHUB_REF])
    subprocess.check_call(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"])
else:
    checkout = Path(LOCAL_REPO)
checkout = checkout.resolve()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{checkout}[training]"])
sys.path.insert(0, str(checkout))


## 3. Mount Drive, load configuration, and verify held-out questions

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

import torch
from adaptive_rag.config import SLM_MODEL, ANSWERING_MODEL, SEED, FINAL_CONTEXT_TOP_K, TEST_SUBSET_PATH, VAL_SUBSET_PATH
from adaptive_rag.data.prepare_data import load_subset

if RUN_BASELINES or RUN_ADAPTIVE or RUN_VALIDATION_CURVE:
    assert torch.cuda.is_available(), "Use a Colab GPU runtime"
TEST_PATH = TEST_SUBSET_PATH if TEST_PATH is None else Path(TEST_PATH)
assert TEST_PATH.is_file(), f"Test subset missing: {TEST_PATH}"
if RUN_ADAPTIVE or RUN_VALIDATION_CURVE:
    assert ADAPTER_DIR.is_dir(), f"Adapter missing: {ADAPTER_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
subset = load_subset(TEST_PATH)
assert len(subset) > 0
ids = [str(row["id"]) for row in subset]
assert len(ids) == len(set(ids)), "Duplicate test question IDs"
print("Configured test path:", TEST_SUBSET_PATH)
print("Actual test path:", TEST_PATH)
print("Questions:", len(subset), "| Seed:", SEED, "| Model:", SLM_MODEL)
print("Answering model:", ANSWERING_MODEL, "| Context top-k:", FINAL_CONTEXT_TOP_K)


In [ ]:
if RUN_BASELINES or RUN_ADAPTIVE or RUN_VALIDATION_CURVE:
    import os
    try:
        from google.colab import userdata
    except ImportError:
        pass  # Local runs use .env or environment variables.
    else:
        # Grant notebook access to the required secrets. Ports/TLS default to 443/true.
        for name in ("WEAVIATE_HTTP_HOST", "WEAVIATE_GRPC_HOST", "WEAVIATE_API_KEY"):
            if not os.getenv(name):
                os.environ[name] = userdata.get(name)
        for name in ("WEAVIATE_HTTP_PORT", "WEAVIATE_HTTP_SECURE", "WEAVIATE_GRPC_PORT", "WEAVIATE_GRPC_SECURE"):
            if name not in os.environ:
                try:
                    os.environ[name] = userdata.get(name)
                except userdata.SecretNotFoundError:
                    pass
    from adaptive_rag.database.weaviate_client import check_retrieval_collection
    print(f"Weaviate ready. Indexed passages: {check_retrieval_collection():,}")


## 3b. Choose the decision threshold on the validation subset

With the strategy prefilled, `adaptive` at threshold tau is exactly `adaptive_split` for questions with P(split) >= tau and `adaptive_once` for the rest. So the quality-cost curve needs only the two forced runs on the validation subset plus per-question P(split) scores. tau is the lowest P(split) among the SPLIT-routed questions at the smallest SPLIT share that reaches `GAIN_FRACTION` of the always-SPLIT recall gain. The test set is never used to choose tau.


In [ ]:
import gc, json
import matplotlib.pyplot as plt
import pandas as pd
from adaptive_rag.evaluation.routing import DEFAULT_BUDGETS, choose_operating_point, routing_curve
from adaptive_rag.evaluation.runner import run_evaluation
from adaptive_rag.planner.adaptive import load_adaptive_planner, score_split_probabilities

THRESHOLD_PATH = RESULTS_DIR / f"{EXPERIMENT_PREFIX}_decision_threshold.json"
VAL_SCORES_PATH = RESULTS_DIR / f"{EXPERIMENT_PREFIX}_val_p_split.json"
TEST_SCORES_PATH = RESULTS_DIR / f"{EXPERIMENT_PREFIX}_test_p_split.json"
VAL_PREFIX = f"{EXPERIMENT_PREFIX}_val"
RECALL_COL = f"recall_at_{FINAL_CONTEXT_TOP_K}"


def score_questions(subset_path, out_path):
    """P(split) per question id from the first strategy-token logits; cached next to the results."""
    if out_path.is_file():
        saved = json.loads(out_path.read_text(encoding="utf-8"))
        assert saved["adapter_dir"] == str(ADAPTER_DIR), f"{out_path} was scored with another adapter; delete it."
        return saved["scores"]
    rows = load_subset(subset_path)
    model, tokenizer = load_adaptive_planner(ADAPTER_DIR, base_model_name=SLM_MODEL, use_4bit=ADAPTIVE_USE_4BIT)
    probs = score_split_probabilities(model, tokenizer, [row["question"] for row in rows])
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    scores = {str(row["id"]): p for row, p in zip(rows, probs)}
    out_path.write_text(json.dumps({"adapter_dir": str(ADAPTER_DIR), "scores": scores}), encoding="utf-8")
    return scores


def read_forced(prefix):
    paths = {s: RESULTS_DIR / f"{prefix}_{s}" / "results.parquet" for s in ("adaptive_once", "adaptive_split")}
    return {s: pd.read_parquet(p) for s, p in paths.items()}


def plot_curve(curve, marks, title):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(curve["learned_pairs"], curve["learned_recall"], marker="o", label="Learned router (P(split) ranking)")
    ax.plot(curve["random_pairs"], curve["random_recall"], ls="--", color="gray", label="Random router (same budget)")
    ax.plot(curve["oracle_pairs"], curve["oracle_recall"], ls=":", color="green", label="Oracle router (upper bound)")
    for label, x, y in marks:
        ax.scatter([x], [y], s=120, marker="*", color="red", zorder=5, label=label)
    ax.set(xlabel="Mean reranked pairs per question (cost)", ylabel=f"Mean {RECALL_COL}", title=title)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


if RUN_VALIDATION_CURVE:
    for strategy in ("adaptive_once", "adaptive_split"):
        run_evaluation(
            experiment_id=f"{VAL_PREFIX}_{strategy}", dataset_path=VAL_SUBSET_PATH,
            planner_strategy=strategy, sampling_seed=SEED, adapter_dir=ADAPTER_DIR,
            adaptive_use_4bit=ADAPTIVE_USE_4BIT, results_dir=RESULTS_DIR,
        )
    val_scores = score_questions(VAL_SUBSET_PATH, VAL_SCORES_PATH)
    forced = read_forced(VAL_PREFIX)
    val_curve = routing_curve(forced["adaptive_once"], forced["adaptive_split"], val_scores, recall_col=RECALL_COL)
    point = choose_operating_point(val_curve, GAIN_FRACTION)
    realized = sum(p >= point["tau"] for p in val_scores.values()) / len(val_scores)
    display(val_curve.round(4))
    val_curve.to_csv(RESULTS_DIR / f"{EXPERIMENT_PREFIX}_val_routing_curve.csv", index=False)
    plot_curve(val_curve, [(f"Chosen: tau={point['tau']:.3f}", val_curve.loc[val_curve["budget"] == point["budget"], "learned_pairs"].iloc[0], point["recall"])],
               "Validation: recall vs. cost by SPLIT share")
    THRESHOLD_PATH.write_text(json.dumps({
        "threshold": point["tau"], "budget": point["budget"], "gain_fraction": GAIN_FRACTION,
        "total_recall_gain_always_split": point["total_gain"], "val_recall_at_threshold": point["recall"],
        "val_pairs_at_threshold": point["pairs"], "n_val": len(val_scores), "realized_val_split_rate": realized,
        "adapter_dir": str(ADAPTER_DIR), "val_path": str(VAL_SUBSET_PATH),
    }, indent=2), encoding="utf-8")
    print(f"Chosen tau = {point['tau']:.4f}: {point['budget']:.1%} of validation questions to SPLIT "
          f"(realized {realized:.1%}), capturing {GAIN_FRACTION:.0%} of the always-SPLIT recall gain.")
    print("Saved:", THRESHOLD_PATH)

DECISION_THRESHOLD = None
if THRESHOLD_PATH.is_file():
    saved_threshold = json.loads(THRESHOLD_PATH.read_text(encoding="utf-8"))
    assert saved_threshold["adapter_dir"] == str(ADAPTER_DIR), "Threshold was chosen for another adapter; rerun the validation curve."
    DECISION_THRESHOLD = saved_threshold["threshold"]
    print("Decision threshold tau:", DECISION_THRESHOLD)
else:
    print("No threshold file yet. Set RUN_VALIDATION_CURVE=True before running the adaptive condition.")


## 4. Evaluate six conditions using `run_evaluation()`

Set `RUN_BASELINES` and `RUN_ADAPTIVE` independently. Use the same held-out test questions and retrieval settings for every run. Invalid planner outputs are saved with `planner_status=failed`; metric means exclude these rows, so compare success rates too.


In [ ]:
from adaptive_rag.evaluation.runner import run_evaluation

BASELINES = ("always_once", "always_split", "prompted")
ADAPTIVE_MODES = ("adaptive_once", "adaptive_split", "adaptive")
STRATEGIES = (*BASELINES, *ADAPTIVE_MODES)
if RUN_ADAPTIVE:
    assert DECISION_THRESHOLD is not None, "Run the validation curve first (RUN_VALIDATION_CURVE=True) to create the threshold file."

for strategy in STRATEGIES:
    is_finetuned = strategy in ADAPTIVE_MODES
    enabled = RUN_ADAPTIVE if is_finetuned else RUN_BASELINES
    if not enabled:
        print(f"Skipped {strategy}")
        continue
    run_evaluation(
        experiment_id=f"{EXPERIMENT_PREFIX}_{strategy}",
        dataset_path=TEST_PATH,
        planner_strategy=strategy,
        sampling_seed=SEED,
        adapter_dir=ADAPTER_DIR if is_finetuned else None,
        adaptive_use_4bit=ADAPTIVE_USE_4BIT if is_finetuned else False,
        decision_threshold=DECISION_THRESHOLD if strategy == "adaptive" else None,
        results_dir=RESULTS_DIR,
    )


## 6. Compare results

Only experiments with saved `results.parquet` are included. Verify all have exactly the same test IDs before interpreting comparisons. The table reports mean metrics and costs, not statistical significance.

In [ ]:
import pandas as pd

strategies = STRATEGIES
frames = {}
for strategy in strategies:
    path = RESULTS_DIR / f"{EXPERIMENT_PREFIX}_{strategy}" / "results.parquet"
    if path.is_file():
        frames[strategy] = pd.read_parquet(path)
    else:
        print("Not found:", path)

if frames:
    expected = set(ids)
    for strategy, df in frames.items():
        assert "id" in df.columns, f"Missing id column: {strategy}"
        assert len(df) == len(expected) and set(df["id"].astype(str)) == expected, (
            f"{strategy} results do not match the complete held-out test subset"
        )
    metric_cols = [
        f"recall_at_{FINAL_CONTEXT_TOP_K}", f"ndcg_at_{FINAL_CONTEXT_TOP_K}",
        "answer_em", "answer_f1", "search_calls", "embedding_calls",
        "reranker_calls", "reranked_pairs", "rewrite_calls",
    ]
    comparison = pd.DataFrame({
        name: {metric: float(df[metric].mean()) for metric in metric_cols}
        for name, df in frames.items()
    }).T
    comparison["planner_success_rate"] = {
        name: float((df["planner_status"] == "ok").mean()) if "planner_status" in df else 1.0
        for name, df in frames.items()
    }
    comparison["planner_retries"] = {
        name: int((df["planner_attempts"] > 1).sum()) if "planner_attempts" in df else 0
        for name, df in frames.items()
    }
    print("Metric means exclude planner failures (NaN). Check success rates alongside them.")
    display(comparison.round(3))
    comparison.to_csv(RESULTS_DIR / f"{EXPERIMENT_PREFIX}_comparison.csv")
else:
    print("No completed experiments found yet.")


In [ ]:
# Test-set routing curve: the same stitching as on validation, with the validation-chosen tau marked.
forced_paths = [RESULTS_DIR / f"{EXPERIMENT_PREFIX}_{s}" / "results.parquet" for s in ("adaptive_once", "adaptive_split")]
if DECISION_THRESHOLD is not None and all(p.is_file() for p in forced_paths):
    test_scores = score_questions(TEST_PATH, TEST_SCORES_PATH)
    forced = read_forced(EXPERIMENT_PREFIX)
    realized = sum(p >= DECISION_THRESHOLD for p in test_scores.values()) / len(test_scores)
    test_curve = routing_curve(forced["adaptive_once"], forced["adaptive_split"], test_scores,
                               budgets=sorted({*DEFAULT_BUDGETS, realized}), recall_col=RECALL_COL)
    at_tau = test_curve.loc[(test_curve["budget"] - realized).abs().idxmin()]
    display(test_curve.round(4))
    test_curve.to_csv(RESULTS_DIR / f"{EXPERIMENT_PREFIX}_test_routing_curve.csv", index=False)
    plot_curve(test_curve, [(f"Validation tau={DECISION_THRESHOLD:.3f}", at_tau["learned_pairs"], at_tau["learned_recall"])],
               "Test: recall vs. cost by SPLIT share")
    print(f"At tau={DECISION_THRESHOLD:.4f}, {realized:.1%} of test questions go to SPLIT "
          f"(stitched recall {at_tau['learned_recall']:.4f}, {at_tau['learned_pairs']:.1f} reranked pairs).")
    if "adaptive" in frames:
        print(f"Measured adaptive run: recall {frames['adaptive'][RECALL_COL].mean():.4f}, "
              f"{frames['adaptive']['reranked_pairs'].mean():.1f} reranked pairs (should match closely).")
else:
    print("Test routing curve skipped: need the threshold file and both adaptive_once and adaptive_split test results.")


## 7. Strategy mix and paired differences

A paired comparison is appropriate because the same questions are used in every experiment. These descriptive differences do not constitute statistical significance testing.

In [ ]:
if frames:
    for name, df in frames.items():
        print(name, "strategy counts:", df["strategy"].value_counts().to_dict())
        if "planner_status" in df:
            print("  planner failures:", int((df["planner_status"] == "failed").sum()))
    if "adaptive" in frames:
        adaptive = frames["adaptive"].set_index("id")
        for baseline in (*BASELINES, "adaptive_once", "adaptive_split"):
            if baseline not in frames:
                continue
            reference = frames[baseline].set_index("id").loc[adaptive.index]
            print(f"Fine-tuned adaptive minus {baseline} (paired successful cases):")
            for metric in [f"recall_at_{FINAL_CONTEXT_TOP_K}", f"ndcg_at_{FINAL_CONTEXT_TOP_K}", "answer_f1", "search_calls"]:
                diff = (adaptive[metric] - reference[metric]).mean()
                print(f"  {metric}: {diff:+.4f}")


## Outputs and interpretation

Six experiment folders, each containing `config.json`, `results.parquet`, `results.csv`, and `summary.json`. The central comparison is `prompted` versus `adaptive`; forced conditions diagnose query generation and strategy selection. The routing curves (validation and test) compare the learned router with random and oracle routing at equal SPLIT share; a router that sits on the random line has learned nothing worth its cost. The notebook has not been GPU-tested.
